# Extração CNES — Cadastro Nacional de Estabelecimentos de Saúde · Conecta Saúde

Este notebook documenta a extração da fonte **CNES**, que fornece a **capacidade instalada** do sistema de saúde: estabelecimentos e leitos.

No Índice Composto de Pressão Assistencial (ICPA), o CNES é o **denominador** — é ele que responde "quantos leitos existem" contra o "quantas internações aconteceram" que vem do SIH/SUS.

---

## 1. Papel do CNES no projeto

O CNES será usado para responder:

- quantos leitos SUS existem por município e por competência;
- quantos estabelecimentos com leito existem em cada município;
- qual o tipo, a natureza jurídica e a gestão de cada unidade.

Cruzamentos previstos:

| Cruzamento | Chave | Indicador resultante |
|---|---|---|
| CNES × SIH/SUS | `CNES` | Internações por leito SUS |
| CNES × IBGE | Município | Leitos SUS por 10 mil habitantes |

### Os dois grupos usados

| Grupo | Conteúdo | Competências extraídas |
|---|---|---|
| `LT` | Leitos por estabelecimento, tipo e especialidade | **12** — leito abre e fecha ao longo do ano, e o arquivo é pequeno |
| `ST` | Atributos do estabelecimento | **1** — tipo e gestão são praticamente estáticos |

Os grupos `PF`, `EQ`, `SR` e `HB` ficam fora do escopo do MVP.

## 2. Endpoint público

Estrutura dos arquivos no servidor FTP do DATASUS:

```text
ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/{GRUPO}/{GRUPO}{UF}{AAMM}.dbc
```

Exemplos confirmados por listagem do diretório:

```text
/dissemin/publicos/CNES/200508_/Dados/LT/LTSP2412.dbc   (100.095 bytes)
/dissemin/publicos/CNES/200508_/Dados/ST/STSP2412.dbc
```

O login é anônimo: usuário `anonymous`, sem senha.

## 3. Parâmetros da extração

In [ ]:
from ftplib import FTP
from pathlib import Path
import struct
import pandas as pd

TODAS_AS_UFS = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG", "MS",
    "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR", "RS", "SC",
    "SE", "SP", "TO",
]

UFS = TODAS_AS_UFS

ANO = 2024
MESES_LT = list(range(1, 13))   # leitos: janela completa
MES_ST = 12                     # estabelecimentos: competência de referência

FTP_HOST = "ftp.datasus.gov.br"
DIR_CNES = "/dissemin/publicos/CNES/200508_/Dados"


def raiz_do_projeto() -> Path:
    '''Devolve a pasta do repositório, subindo até encontrar o .git.

    Sem isto os caminhos dependeriam de onde o Jupyter foi aberto: rodando a
    partir de "Fontes de dados" os dados cairiam dentro dessa pasta, e a
    partir da raiz cairiam em outro lugar.
    '''
    atual = Path.cwd().resolve()
    for pasta in (atual, *atual.parents):
        if (pasta / ".git").exists():
            return pasta
    return atual


RAIZ = raiz_do_projeto()
DIRETORIO_RAW = RAIZ / "dados" / "raw" / "cnes"
DIRETORIO_TRATADO = RAIZ / "dados" / "tratado" / "cnes"
DIRETORIO_REFERENCIA = RAIZ / "referencias" / "cnes"

for pasta in (DIRETORIO_RAW, DIRETORIO_TRATADO, DIRETORIO_REFERENCIA):
    pasta.mkdir(parents=True, exist_ok=True)


def nome_arquivo(grupo: str, uf: str, ano: int, mes: int) -> str:
    return f"{grupo.upper()}{uf.upper()}{str(ano)[-2:]}{str(mes).zfill(2)}.dbc"


print(f"Raiz do projeto : {RAIZ}")
print(f"UFs             : {len(UFS)} ({', '.join(UFS[:6])}{'...' if len(UFS) > 6 else ''})")
print(f"Arquivos LT     : {len(UFS) * len(MESES_LT)}")

Raiz do projeto : C:\Users\Vitor Nobre\Documents\Workspace\conecta-saude
UFs             : 27 (AC, AL, AM, AP, BA, CE...)
Arquivos LT     : 324


## 4. Conexão FTP e verificação de disponibilidade

A função abaixo lista o diretório do grupo e conta quantas competências existem por UF.

O modo passivo é ligado explicitamente: com firewall ou NAT entre a máquina e a internet, o modo ativo do FTP costuma falhar na conexão de dados.

In [ ]:
def conectar_ftp() -> FTP:
    ftp = FTP(FTP_HOST, timeout=300)
    ftp.login()          # anônimo
    ftp.set_pasv(True)   # obrigatório atrás de firewall/NAT
    return ftp


def conferir_disponibilidade(grupo: str, ufs: list[str], ano: int) -> dict[str, int]:
    '''Conta as competências publicadas de cada UF, numa única listagem.

    Uma listagem só para todas as UFs: pedir uma por UF seriam 27 idas ao
    servidor para responder a mesma pergunta.
    '''
    ftp = conectar_ftp()
    try:
        ftp.cwd(f"{DIR_CNES}/{grupo.upper()}")
        publicados = set(ftp.nlst())
    finally:
        ftp.quit()

    return {
        uf: sum(1 for a in publicados
                if a.startswith(f"{grupo.upper()}{uf.upper()}{str(ano)[-2:]}"))
        for uf in ufs
    }


disponiveis_lt = conferir_disponibilidade("LT", UFS, ANO)
disponiveis_st = conferir_disponibilidade("ST", UFS, ANO)

print(f"LT — competências encontradas por UF: {sorted(set(disponiveis_lt.values()))}")
print(f"ST — competências encontradas por UF: {sorted(set(disponiveis_st.values()))}")

incompletas = {uf: n for uf, n in disponiveis_lt.items() if n < len(MESES_LT)}
if incompletas:
    print(f"\nATENÇÃO — UFs com menos de {len(MESES_LT)} competências no LT: {incompletas}")
else:
    print(f"\nTodas as {len(UFS)} UFs têm as {len(MESES_LT)} competências do LT publicadas.")

LT — competências encontradas por UF: [12]
ST — competências encontradas por UF: [12]

Todas as 27 UFs têm as 12 competências do LT publicadas.


## 5. Download e conversão

O `.dbc` é um dBase comprimido com algoritmo proprietário do DATASUS. O fluxo é sempre o mesmo:

```text
.dbc  →  datasus_dbc.decompress  →  .dbf  →  dbfread  →  DataFrame
```

A função apaga os arquivos intermediários assim que o DataFrame existe, para não encher o disco.


In [ ]:
import datasus_dbc
from dbfread import DBF


def baixar_arquivo(grupo: str, arquivo: str, ftp: FTP | None = None) -> Path:
    '''Baixa um .dbc do FTP. Reaproveita o arquivo em disco se já existir.

    Passando uma conexão já aberta em `ftp`, ela é reutilizada — é o que torna
    a extração de várias UFs 2,4x mais rápida.
    '''
    destino = DIRETORIO_RAW / arquivo
    if destino.exists() and destino.stat().st_size > 0:
        return destino

    propria = ftp is None      # abri aqui, então fecho aqui
    if propria:
        ftp = conectar_ftp()
        ftp.cwd(f"{DIR_CNES}/{grupo.upper()}")
    try:
        with open(destino, "wb") as f:
            ftp.retrbinary(f"RETR {arquivo}", f.write, blocksize=65536)
    finally:
        if propria:
            ftp.quit()
    return destino


def corrigir_terminador_dbf(caminho_dbf: Path) -> bool:
    '''Grava o 0x0D que fecha a lista de campos, quando o DATASUS o omite.

    O cabeçalho do DBF declara seu próprio tamanho, e a lista de campos ocupa
    32 bytes por campo mais 1 byte de terminador. Logo o 0x0D tem de estar em
    `tamanho_do_cabecalho - 1`, e escrever ali não toca em nenhum dado.
    '''
    with open(caminho_dbf, "r+b") as arquivo:
        tam_cabecalho, = struct.unpack("<H", arquivo.read(12)[8:10])

        if (tam_cabecalho - 33) % 32 != 0:
            raise ValueError(
                f"{caminho_dbf.name}: cabeçalho de {tam_cabecalho} bytes não "
                "corresponde a campos de 32 bytes — layout inesperado"
            )

        arquivo.seek(tam_cabecalho - 1)
        if arquivo.read(1) == b"\x0d":
            return False
        arquivo.seek(tam_cabecalho - 1)
        arquivo.write(b"\x0d")
        return True


def ler_dbc(caminho_dbc: Path, limpar: bool = True) -> pd.DataFrame:
    '''Converte .dbc em DataFrame, apagando os intermediários.'''
    caminho_dbf = caminho_dbc.with_suffix(".dbf")
    datasus_dbc.decompress(str(caminho_dbc), str(caminho_dbf))
    corrigir_terminador_dbf(caminho_dbf)

    tabela = DBF(str(caminho_dbf), encoding="latin-1", load=False)
    declaradas = len(tabela)
    df = pd.DataFrame(iter(tabela))

    if len(df) != declaradas:
        raise ValueError(f"{caminho_dbc.name}: li {len(df)} linhas de {declaradas} declaradas")

    if limpar:
        caminho_dbf.unlink(missing_ok=True)
        caminho_dbc.unlink(missing_ok=True)
    return df


def extrair_grupo(
    grupo: str,
    ufs: list[str],
    ano: int,
    meses: list[int],
    colunas: list[str] | None = None,
) -> pd.DataFrame:
    '''Extrai um grupo para várias UFs e competências, numa só conexão.

    `colunas` recorta cada arquivo antes de concatenar. Serve ao ST, cujas
    208 colunas ocupariam 795 MB no Brasil inteiro contra 30 MB recortando.
    '''
    frames, falhas, linhas = [], [], 0

    ftp = conectar_ftp()
    ftp.cwd(f"{DIR_CNES}/{grupo.upper()}")
    try:
        for uf in ufs:
            for mes in meses:
                arquivo = nome_arquivo(grupo, uf, ano, mes)
                try:
                    df = ler_dbc(baixar_arquivo(grupo, arquivo, ftp=ftp))
                    if colunas is not None:
                        df = df[[c for c in colunas if c in df.columns]].copy()
                    df["UF"] = uf.upper()
                    df["COMPETENCIA"] = f"{ano}{str(mes).zfill(2)}"
                    df["GRUPO_CNES"] = grupo.upper()
                    frames.append(df)
                    linhas += len(df)
                except Exception as erro:
                    falhas.append((arquivo, f"{type(erro).__name__}: {erro}"))
                    try:
                        ftp.quit()
                    except Exception:
                        pass
                    ftp = conectar_ftp()
                    ftp.cwd(f"{DIR_CNES}/{grupo.upper()}")
            print(f"  {uf}: {linhas:,} linhas acumuladas")
    finally:
        try:
            ftp.quit()
        except Exception:
            pass

    if falhas:
        print(f"\n{len(falhas)} arquivo(s) falharam:")
        for arquivo, motivo in falhas:
            print(f"  {arquivo}: {motivo}")

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


print(f"Extraindo LT: {len(UFS)} UF(s) x {len(MESES_LT)} competência(s)")
cnes_lt_bruto = extrair_grupo("LT", UFS, ANO, MESES_LT)
print(f"\nTotal LT: {cnes_lt_bruto.shape[0]:,} linhas e {cnes_lt_bruto.shape[1]} colunas")
print(f"Memória  : {cnes_lt_bruto.memory_usage(deep=True).sum() / 1024**2:.0f} MB")
cnes_lt_bruto.head()

Extraindo LT: 27 UF(s) x 12 competência(s)
  AC: 3,316 linhas acumuladas
  AL: 12,024 linhas acumuladas
  AM: 20,820 linhas acumuladas
  AP: 22,278 linhas acumuladas
  BA: 68,753 linhas acumuladas
  CE: 93,515 linhas acumuladas
  DF: 100,293 linhas acumuladas
  ES: 112,539 linhas acumuladas
  GO: 147,729 linhas acumuladas
  MA: 169,025 linhas acumuladas
  MG: 231,629 linhas acumuladas
  MS: 244,135 linhas acumuladas
  MT: 259,442 linhas acumuladas
  PA: 283,235 linhas acumuladas
  PB: 297,332 linhas acumuladas
  PE: 324,245 linhas acumuladas
  PI: 335,461 linhas acumuladas
  PR: 374,552 linhas acumuladas
  RJ: 422,444 linhas acumuladas
  RN: 433,765 linhas acumuladas
  RO: 442,227 linhas acumuladas
  RR: 443,861 linhas acumuladas
  RS: 480,105 linhas acumuladas
  SC: 503,674 linhas acumuladas
  SE: 507,812 linhas acumuladas
  SP: 608,749 linhas acumuladas
  TO: 615,693 linhas acumuladas

Total LT: 615,693 linhas e 31 colunas
Memória  : 191 MB


,CNES,CODUFMUN,REGSAUDE,MICR_REG,DISTRSAN,DISTRADM,TPGESTAO,PF_PJ,CPF_CNPJ,NIV_DEP,...,CODLEITO,QT_EXIST,QT_CONTR,QT_SUS,QT_NSUS,COMPETEN,NAT_JUR,UF,COMPETENCIA,GRUPO_CNES
0,5701929,120001,001,,,,E,3,00000000000000,3,...,33,3,0,3,0,202401,1023,AC,202401,LT
1,5701929,120001,001,,,,E,3,00000000000000,3,...,43,1,0,1,0,202401,1023,AC,202401,LT
2,5701929,120001,001,,,,E,3,00000000000000,3,...,45,2,0,2,0,202401,1023,AC,202401,LT
3,5701929,120001,001,,,,E,3,00000000000000,3,...,47,1,0,1,0,202401,1023,AC,202401,LT
4,2001020,120005,002,,,,E,3,04034526001034,3,...,33,7,0,7,0,202401,1023,AC,202401,LT


## 6. Colunas disponíveis

O layout do CNES varia entre competências. 

In [4]:
print("Colunas do LT:")
print(cnes_lt_bruto.columns.tolist())

Colunas do LT:
['CNES', 'CODUFMUN', 'REGSAUDE', 'MICR_REG', 'DISTRSAN', 'DISTRADM', 'TPGESTAO', 'PF_PJ', 'CPF_CNPJ', 'NIV_DEP', 'CNPJ_MAN', 'ESFERA_A', 'ATIVIDAD', 'RETENCAO', 'NATUREZA', 'CLIENTEL', 'TP_UNID', 'TURNO_AT', 'NIV_HIER', 'TERCEIRO', 'TP_LEITO', 'CODLEITO', 'QT_EXIST', 'QT_CONTR', 'QT_SUS', 'QT_NSUS', 'COMPETEN', 'NAT_JUR', 'UF', 'COMPETENCIA', 'GRUPO_CNES']


## 7. Seleção de campos e tratamento de tipos

Três cuidados obrigatórios:

**`CNES` e `CODUFMUN` como texto.** O `CNES` tem 7 posições e o `CODUFMUN` tem 6, ambos podendo ter zero à esquerda. Se o pandas ler como inteiro, o zero some e o join com o SIH falha em parte dos registros — sem erro, apenas com linhas perdidas.

**Quantidades como número.** Se `QT_SUS` vier como texto, o `sum()` concatena strings em vez de somar, e o resultado sai absurdo sem sinal de erro.

In [5]:
CANDIDATAS_LT = [
    "CNES", "CODUFMUN", "UF", "TP_LEITO", "CODLEITO",
    "QT_EXIST", "QT_CONTR", "QT_SUS", "QT_NSUS", "COMPETEN",
    "COMPETENCIA", "GRUPO_CNES",
]

def selecionar_colunas(df: pd.DataFrame, candidatas: list[str]) -> pd.DataFrame:
    presentes = [c for c in candidatas if c in df.columns]
    ausentes = [c for c in candidatas if c not in df.columns]
    if ausentes:
        print(f"Colunas ausentes neste layout: {ausentes}")
    return df[presentes].copy()


def tratar_tipos_lt(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ["CNES", "CODUFMUN", "TP_LEITO", "CODLEITO"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    if "CNES" in df.columns:
        df["CNES"] = df["CNES"].str.zfill(7)
    if "CODUFMUN" in df.columns:
        df["CODUFMUN"] = df["CODUFMUN"].str.zfill(6)
    for col in ["QT_EXIST", "QT_CONTR", "QT_SUS", "QT_NSUS"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
    return df


cnes_lt = tratar_tipos_lt(selecionar_colunas(cnes_lt_bruto, CANDIDATAS_LT))
cnes_lt.info()

<class 'pandas.DataFrame'>
RangeIndex: 615693 entries, 0 to 615692
Data columns (total 12 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   CNES         615693 non-null  str  
 1   CODUFMUN     615693 non-null  str  
 2   UF           615693 non-null  str  
 3   TP_LEITO     615693 non-null  str  
 4   CODLEITO     615693 non-null  str  
 5   QT_EXIST     615693 non-null  int64
 6   QT_CONTR     615693 non-null  int64
 7   QT_SUS       615693 non-null  int64
 8   QT_NSUS      615693 non-null  int64
 9   COMPETEN     615693 non-null  str  
 10  COMPETENCIA  615693 non-null  str  
 11  GRUPO_CNES   615693 non-null  str  
dtypes: int64(4), str(8)
memory usage: 75.5 MB


### 7.1 Rótulos dos códigos — as tabelas de domínio do CNES

`TP_LEITO` e `CODLEITO` chegam como números, e número sozinho não sustenta decisão analítica. As descrições não estão nos arquivos de dados: ficam nas **tabelas de conversão do TabNet**, os arquivos `.CNV`, publicados no mesmo servidor FTP:

```text
ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Auxiliar/TAB_CNES.zip
```

São exatamente as tabelas que o DATASUS usa para rotular as próprias tabulações públicas. 

#### Um código que o CNV ainda não cobre

O `CODLEITO` **96** aparece nos dados de 16 UFs — 1.010 linhas, 2.526 leitos SUS — e não existe na versão atual do `Esp_leit.CNV`, que vai até o 95. É leito complementar (`TP_LEITO` 3), provavelmente criado depois da última atualização da tabela.

São 0,06% dos leitos SUS do país, mas a célula seguinte reporta o caso em vez de silenciá-lo: código sem rótulo vira `(não catalogado no CNV)` e aparece no resumo, com a quantidade de leitos envolvida. Um `merge` que deixasse isso virar `NaN` esconderia, quando o CNV ficasse defasado, exatamente o tipo de leito novo que mais interessaria notar.

In [ ]:
import zipfile

DIR_AUXILIAR = "/dissemin/publicos/CNES/200508_/Auxiliar"
ARQ_TABELAS = "TAB_CNES.zip"

# Quem diz qual CNV pertence a qual coluna são os arquivos .def do próprio
# pacote: Leitos_Especialidade.def para o LT, Estabelecimento.def para o ST.
CNV_LT = {
    "TP_LEITO": "CNV/tip1leit.cnv",
    "CODLEITO": "CNV/Esp_leit.CNV",
}
CNV_ST = {
    "TP_UNID": "CNV/TP_ESTAB.CNV",
    "TPGESTAO": "CNV/TPGESTAO.CNV",
    "NAT_JUR": "CNV/NATJURC.CNV",
    "ATIVIDAD": "CNV/Ativ_Ens.CNV",
    "NIV_DEP": "CNV/NIVELDEP.CNV",
    "TURNO_AT": "CNV/TurnosAt.cnv",
    "VINC_SUS": "CNV/Vinc_SUS.cnv",
    "CLIENTEL": "CNV/Flux_Cli.CNV",
}
TODOS_OS_CNV = {**CNV_LT, **CNV_ST}


def baixar_tabelas_dominio(nomes: list[str]) -> dict[str, bytes]:
    """Baixa o pacote de tabelas do CNES e guarda só os CNV pedidos.

    Mesmo fluxo do `ler_dbc`: baixa, extrai o que interessa, apaga o
    intermediário. O zip tem 121 MB e os dez CNV somam poucos KB, então o que
    fica no disco é só o cache — e da segunda execução em diante nem o
    download acontece.
    """
    caminhos = {n: DIRETORIO_REFERENCIA / n.replace("/", "_") for n in nomes}
    if all(c.exists() and c.stat().st_size > 0 for c in caminhos.values()):
        print(f"{len(caminhos)} CNV já em cache local — nenhum download necessário.")
        return {n: c.read_bytes() for n, c in caminhos.items()}

    zip_local = DIRETORIO_RAW / ARQ_TABELAS
    print(f"Baixando {ARQ_TABELAS} (~121 MB, uma vez só)...")
    ftp = conectar_ftp()
    try:
        ftp.cwd(DIR_AUXILIAR)
        with open(zip_local, "wb") as arquivo:
            ftp.retrbinary(f"RETR {ARQ_TABELAS}", arquivo.write, blocksize=65536)
    finally:
        ftp.quit()

    with zipfile.ZipFile(zip_local) as pacote:
        # O zip mistura caixas nos nomes (Vinc_SUS.cnv, TP_ESTAB.CNV),
        # então a busca ignora maiúsculas e minúsculas.
        reais = {n.lower(): n for n in pacote.namelist()}
        for nome, caminho in caminhos.items():
            real = reais.get(nome.lower())
            if real is None:
                raise KeyError(f"{nome} não está em {ARQ_TABELAS}")
            caminho.write_bytes(pacote.read(real))
            print(f"  extraído {real}: {caminho.stat().st_size} bytes")

    zip_local.unlink()   # o pacote não serve para mais nada
    return {n: c.read_bytes() for n, c in caminhos.items()}


def expandir_faixa(faixa: str, largura: int) -> list[str]:
    """Converte a faixa de um CNV na lista de códigos que ela cobre.

    O campo aceita três formas, e tratar só a primeira perderia domínios
    inteiros em silêncio:

        "05"          código simples
        "Z,S"         lista separada por vírgula
        "1000-1999"   intervalo numérico
    """
    codigos = []
    for parte in faixa.split(","):
        parte = parte.strip()
        if not parte:
            continue
        inicio, traco, fim = parte.partition("-")
        if traco and inicio.isdigit() and fim.isdigit():
            codigos.extend(str(n).zfill(largura) for n in range(int(inicio), int(fim) + 1))
        elif traco and not inicio:
            # "-99" e "-zzzz" são marcadores de "não informado" do TabWin,
            # não intervalos abertos.
            codigos.append(fim.zfill(largura) if fim.isdigit() else fim)
        else:
            codigos.append(parte.zfill(largura) if parte.isdigit() else parte)
    return codigos


def ler_cnv(conteudo: bytes, col_codigo: str, col_descricao: str) -> pd.DataFrame:
    """Converte um .CNV do TabWin em DataFrame de código e descrição.

    Cada linha tem número de ordem, descrição e código, nesta ordem:

        "      3  CIRURGIA GERAL                          03"
         ^ordem   ^descrição                              ^código

    O código é sempre o último campo, e é por isso que o corte é pelo último
    espaço (`rsplit`): algumas descrições encostam no código, e um `split`
    comum perderia justamente essas linhas.
    """
    linhas = conteudo.decode("latin-1").splitlines()

    # O cabeçalho vem como "70 2 L" ou "N 45 2 L" — alguns arquivos trazem uma
    # letra na frente. Filtrar só os tokens numéricos evita ler a contagem de
    # categorias no lugar da largura; com a largura errada, o zfill destruiria
    # todos os códigos sem levantar erro.
    numeros = [t for t in linhas[0].split() if t.isdigit()]
    largura = int(numeros[1])

    registros = []
    for linha in linhas[1:]:
        if not linha.strip():
            continue
        miolo, faixa = linha.rsplit(None, 1)
        _, descricao = miolo.split(None, 1)
        for codigo in expandir_faixa(faixa, largura):
            registros.append({col_codigo: codigo, col_descricao: descricao.strip()})

    return pd.DataFrame(registros).drop_duplicates(subset=[col_codigo])


cnvs = baixar_tabelas_dominio(list(TODOS_OS_CNV.values()))

# Um DataFrame de domínio por coluna, com a descrição nomeada de forma previsível.
dominios = {
    coluna: ler_cnv(cnvs[caminho], coluna, f"{coluna}_DESC")
    for coluna, caminho in TODOS_OS_CNV.items()
}

# O rótulo do tipo de leito vem como "1-Cirúrgico"; o prefixo só repete o código.
dominios["TP_LEITO"]["TP_LEITO_DESC"] = (
    dominios["TP_LEITO"]["TP_LEITO_DESC"].str.split("-", n=1).str[-1].str.strip()
)

# Nomes que o restante do notebook já usa para os domínios de leito.
dominio_tp_leito = dominios["TP_LEITO"].rename(columns={"TP_LEITO_DESC": "TIPO_LEITO_DESC"})
dominio_codleito = dominios["CODLEITO"].rename(columns={"CODLEITO_DESC": "ESPECIALIDADE_DESC"})

print()
for coluna, df in dominios.items():
    print(f"  {coluna:<10} {len(df):>5} códigos")
display(dominio_tp_leito)

In [ ]:

cnes_lt["CODLEITO"] = cnes_lt["CODLEITO"].str.zfill(2)

cnes_lt = cnes_lt.drop(columns=["TIPO_LEITO_DESC", "ESPECIALIDADE_DESC"], errors="ignore")

antes = len(cnes_lt)
cnes_lt = (
    cnes_lt
    .merge(dominio_tp_leito, on="TP_LEITO", how="left")
    .merge(dominio_codleito, on="CODLEITO", how="left")
)

assert len(cnes_lt) == antes, f"merge multiplicou linhas: {antes:,} -> {len(cnes_lt):,}"
print(f"Linhas preservadas no merge: {len(cnes_lt):,}")

for col_desc, col_cod in [("TIPO_LEITO_DESC", "TP_LEITO"), ("ESPECIALIDADE_DESC", "CODLEITO")]:
    orfaos = cnes_lt[cnes_lt[col_desc].isna()]
    if orfaos.empty:
        print(f"{col_cod}: todos os códigos têm descrição.")
    else:
        print(f"\n{col_cod}: {orfaos[col_cod].nunique()} código(s) sem descrição no CNV —")
        print(orfaos.groupby(col_cod)[["QT_EXIST", "QT_SUS"]].sum().to_string())
    cnes_lt[col_desc] = cnes_lt[col_desc].fillna("(não catalogado no CNV)")

display(cnes_lt.head())

Linhas preservadas no merge: 615,693
TP_LEITO: todos os códigos têm descrição.

CODLEITO: 1 código(s) sem descrição no CNV —
          QT_EXIST  QT_SUS
CODLEITO                  
96            6555    2526


,CNES,CODUFMUN,UF,TP_LEITO,CODLEITO,QT_EXIST,QT_CONTR,QT_SUS,QT_NSUS,COMPETEN,COMPETENCIA,GRUPO_CNES,TIPO_LEITO_DESC,ESPECIALIDADE_DESC
0,5701929,120001,AC,2,33,3,0,3,0,202401,202401,LT,Clínico,CLINICA GERAL
1,5701929,120001,AC,4,43,1,0,1,0,202401,202401,LT,Obstétrico,OBSTETRICIA CLINICA
2,5701929,120001,AC,5,45,2,0,2,0,202401,202401,LT,Pediátrico,PEDIATRIA CLINICA
3,5701929,120001,AC,6,47,1,0,1,0,202401,202401,LT,Outras Especialidades,PSIQUIATRIA
4,2001020,120005,AC,2,33,7,0,7,0,202401,202401,LT,Clínico,CLINICA GERAL


## 8. Decisão — quais tipos de leito contar

O `LT` traz **uma linha por estabelecimento, tipo de leito e especialidade**.

### Dupla contagem: verificada e descartada

A suspeita natural é que o leito complementar (UTI) já esteja embutido no total dos leitos clínicos ou cirúrgicos. **Não está**, e há duas evidências independentes:

- a [nota técnica de Recursos Físicos do CNES](http://tabnet.datasus.gov.br/cgi/cnes/NT_RecursosF%C3%ADsicos.htm) registra que "a partir da competência de Janeiro 2010 os dados referentes a leitos Complementares foram retirados da consulta referente a leitos de Internação, passando a constituir uma consulta específica";
- nos dados, nenhum `CODLEITO` aparece em mais de um `TP_LEITO` — as especialidades formam conjuntos disjuntos por tipo (cirúrgicos de `01` a `16`, clínicos de `31` a `49`, complementares de `61` em diante).

Ou seja: cada leito é contado uma única vez. O que resta a decidir não é como corrigir uma soma inflada, e sim **qual recorte de capacidade instalada faz sentido como denominador do ICPA**.

In [ ]:
ultima = cnes_lt["COMPETENCIA"].max()
foto = cnes_lt.query("COMPETENCIA == @ultima")
escopo = "Brasil" if len(UFS) == len(TODAS_AS_UFS) else ", ".join(UFS)

print(f"Competência analisada: {ultima} | escopo: {escopo}\n")

print("Leitos por tipo:")
por_tipo = (
    foto.groupby(["TP_LEITO", "TIPO_LEITO_DESC"])[["QT_EXIST", "QT_SUS"]]
    .sum()
    .sort_values("QT_SUS", ascending=False)
)
por_tipo["%_SUS"] = (100 * por_tipo["QT_SUS"] / por_tipo["QT_EXIST"]).round(1)
display(por_tipo)

print("\nAs 25 maiores especialidades, em leitos SUS:")
display(
    foto.groupby(["TIPO_LEITO_DESC", "ESPECIALIDADE_DESC"])[["QT_EXIST", "QT_SUS"]]
    .sum()
    .sort_values("QT_SUS", ascending=False)
    .head(25)
)

# QT_NSUS é informado pelo CNES e deve ser exatamente QT_EXIST - QT_SUS.
divergencia = int((foto["QT_EXIST"] - foto["QT_SUS"] - foto["QT_NSUS"]).abs().sum())
print(f"\nConferência QT_NSUS = QT_EXIST - QT_SUS: divergência de {divergencia} leitos")

print(f"\nEm {escopo}, competência {ultima}:")
print(f"  leitos existentes : {int(foto['QT_EXIST'].sum()):>9,}")
print(f"  leitos SUS        : {int(foto['QT_SUS'].sum()):>9,}")
print(f"  proporção SUS     : {100 * foto['QT_SUS'].sum() / foto['QT_EXIST'].sum():>8.1f}%")

print("\nEfeito de cada recorte sobre os leitos SUS:")
for rotulo, excluir in [
    ("todos os tipos", []),
    ("sem Hospital/DIA (7)", ["7"]),
    ("sem Hospital/DIA e sem Complementar (7, 3)", ["7", "3"]),
]:
    sub = foto[~foto["TP_LEITO"].isin(excluir)]
    print(f"  {rotulo:<44} {int(sub['QT_SUS'].sum()):>8,} SUS | {int(sub['QT_EXIST'].sum()):>9,} existentes")

if len(UFS) > 1:
    print("\nLeitos SUS por UF:")
    display(
        foto.groupby("UF")[["QT_EXIST", "QT_SUS"]]
        .sum()
        .sort_values("QT_SUS", ascending=False)
    )

Competência analisada: 202412 | escopo: Brasil

Leitos por tipo:


,,QT_EXIST,QT_SUS,%_SUS
TP_LEITO,TIPO_LEITO_DESC,,,
2,Clínico,175187,123500,70.5
1,Cirúrgico,122442,79745,65.1
3,Complementar,80546,43188,53.6
4,Obstétrico,50278,38005,75.6
5,Pediátrico,46975,37029,78.8
6,Outras Especialidades,45768,29073,63.5
7,Hospital/DIA,14370,6544,45.5



As 25 maiores especialidades, em leitos SUS:


,,QT_EXIST,QT_SUS
TIPO_LEITO_DESC,ESPECIALIDADE_DESC,,
Clínico,CLINICA GERAL,137247,100084
Cirúrgico,CIRURGIA GERAL,63086,41781
Pediátrico,PEDIATRIA CLINICA,41111,32940
Obstétrico,OBSTETRICIA CLINICA,25131,20385
Complementar,UTI ADULTO - TIPO II,32229,19409
Obstétrico,OBSTETRICIA CIRURGICA,25147,17620
Outras Especialidades,PSIQUIATRIA,28530,15766
Cirúrgico,ORTOPEDIATRAUMATOLOGIA,19664,15370
Outras Especialidades,CRONICOS,9982,8673



Conferência QT_NSUS = QT_EXIST - QT_SUS: divergência de 0 leitos

Em Brasil, competência 202412:
  leitos existentes :   535,566
  leitos SUS        :   357,084
  proporção SUS     :     66.7%

Efeito de cada recorte sobre os leitos SUS:
  todos os tipos                                357,084 SUS |   535,566 existentes
  sem Hospital/DIA (7)                          350,540 SUS |   521,196 existentes
  sem Hospital/DIA e sem Complementar (7, 3)    307,352 SUS |   440,650 existentes

Leitos SUS por UF:


,QT_EXIST,QT_SUS
UF,,
SP,114445,64217
MG,50729,33441
BA,35744,27406
RJ,46117,24552
RS,33848,22871
PR,31713,21898
PE,26670,18335
CE,22142,17185
MA,16862,14295


### Aplicação do filtro

Os três recortes possíveis, em leitos SUS na competência 2024/12:

<div style="display: flex; justify-content: center;">

| Recorte | Brasil | São Paulo |
|---|---:|---:|
| todos os tipos | 357.084 | 64.217 |
| **sem Hospital/DIA (7)** | **350.540** | **62.092** |
| sem Hospital/DIA e Complementar (7, 3) | — | 52.798 |

</div>

---

> **Decisão registrada:** excluir apenas o tipo **7 — Hospital/DIA**, mantendo os demais, inclusive o tipo 3 (Complementar).
>
> **Hospital/DIA:** É atendimento sem pernoite: o paciente entra e sai no mesmo dia. Não é leito de permanência e por isso não pertence ao denominador de um índice que mede pressão por internação. São 6.544 leitos SUS no Brasil, ou 1,8% do total.
>
> **Complementar:** UTI e unidades intermediárias recebem internação — um paciente pode ser admitido diretamente em UTI e isso gera AIH no SIH/SUS. Como o numerador do ICPA conta essas internações, tirar esses leitos do denominador inflaria artificialmente a pressão medida, sobretudo nos municípios-polo, que concentram a terapia intensiva da região.

In [11]:
# Tipos de leito a EXCLUIR da contagem. Strings, não inteiros: TP_LEITO é texto.
# 7 = Hospital/DIA, atendimento sem pernoite, não é leito de permanência.
TIPOS_EXCLUIR = ["7"]

# Falha cedo se alguém trocar por inteiro ou digitar um tipo inexistente.
assert all(isinstance(t, str) for t in TIPOS_EXCLUIR), "use strings: ['7'], não [7]"
desconhecidos = set(TIPOS_EXCLUIR) - set(dominio_tp_leito["TP_LEITO"])
assert not desconhecidos, f"tipos que não existem no CNV: {desconhecidos}"

if TIPOS_EXCLUIR:
    rotulos = (
        dominio_tp_leito.set_index("TP_LEITO")
        .loc[TIPOS_EXCLUIR, "TIPO_LEITO_DESC"]
        .tolist()
    )
    cnes_lt_filtrado = cnes_lt[~cnes_lt["TP_LEITO"].isin(TIPOS_EXCLUIR)].copy()
    removidos = len(cnes_lt) - len(cnes_lt_filtrado)
    leitos_fora = cnes_lt.loc[cnes_lt["TP_LEITO"].isin(TIPOS_EXCLUIR), "QT_SUS"].sum()
    print(f"Excluídos: {rotulos}")
    print(f"  {removidos:,} registros e {int(leitos_fora):,} leitos SUS fora da contagem")
else:
    cnes_lt_filtrado = cnes_lt.copy()
    print("Nenhum tipo excluído — todos os leitos estão sendo contados.")

print(f"Registros considerados: {len(cnes_lt_filtrado):,}")

Excluídos: ['Hospital/DIA']
  20,862 registros e 76,248 leitos SUS fora da contagem
Registros considerados: 594,831


## 9. Agregação para a camada Silver

Duas saídas, no grão que o restante do pipeline consome:

- **por município e competência** — alimenta o ICPA;
- **por estabelecimento e competência** — alimenta a análise de pressão por hospital.

In [12]:
silver_cnes_municipio = (
    cnes_lt_filtrado.query("QT_SUS > 0")
    .groupby(["CODUFMUN", "COMPETENCIA"])
    .agg(
        leitos_sus=("QT_SUS", "sum"),
        leitos_existentes=("QT_EXIST", "sum"),
        estabelecimentos_com_leito=("CNES", "nunique"),
    )
    .reset_index()
)

silver_cnes_hospital = (
    cnes_lt_filtrado.query("QT_SUS > 0")
    .groupby(["CNES", "CODUFMUN", "COMPETENCIA"])
    .agg(
        leitos_sus=("QT_SUS", "sum"),
        leitos_existentes=("QT_EXIST", "sum"),
    )
    .reset_index()
)

print(f"Município x competência: {len(silver_cnes_municipio):,} linhas")
print(f"Hospital x competência:  {len(silver_cnes_hospital):,} linhas")
display(silver_cnes_municipio.sort_values("leitos_sus", ascending=False).head(15))

Município x competência: 42,715 linhas
Hospital x competência:  73,234 linhas


,CODUFMUN,COMPETENCIA,leitos_sus,leitos_existentes,estabelecimentos_com_leito
30292,355030,202410,17657,20266,147
30293,355030,202411,17657,20266,147
30294,355030,202412,17657,20266,147
30291,355030,202409,17652,20264,146
30290,355030,202408,17641,20291,147
30287,355030,202405,17610,20238,146
30284,355030,202402,17609,20139,145
30288,355030,202406,17607,20242,146
30289,355030,202407,17603,20223,146
30283,355030,202401,17550,20143,145


## 10. Validação de qualidade


In [13]:
validacoes = pd.DataFrame([
    {"verificacao": "Competências extraídas", "valor": cnes_lt["COMPETENCIA"].nunique()},
    {"verificacao": "Municípios distintos", "valor": cnes_lt_filtrado["CODUFMUN"].nunique()},
    {"verificacao": "Estabelecimentos distintos", "valor": cnes_lt_filtrado["CNES"].nunique()},
    {"verificacao": "Estabelecimentos com leito SUS", "valor": int(cnes_lt_filtrado.query("QT_SUS > 0")["CNES"].nunique())},
    {"verificacao": "CNES fora do padrão de 7 dígitos", "valor": int((cnes_lt_filtrado["CNES"].str.len() != 7).sum())},
    {"verificacao": "Município fora do padrão de 6 dígitos", "valor": int((cnes_lt_filtrado["CODUFMUN"].str.len() != 6).sum())},
    {"verificacao": "Registros com QT_SUS igual a zero", "valor": int((cnes_lt_filtrado["QT_SUS"] == 0).sum())},
])

display(validacoes)

print("\nEvolução dos leitos SUS ao longo do ano:")
print(silver_cnes_municipio.groupby("COMPETENCIA")["leitos_sus"].sum())

,verificacao,valor
0,Competências extraídas,12
1,Municípios distintos,3616
2,Estabelecimentos distintos,8686
3,Estabelecimentos com leito SUS,6264
4,CNES fora do padrão de 7 dígitos,0
5,Município fora do padrão de 6 dígitos,0
6,Registros com QT_SUS igual a zero,192760



Evolução dos leitos SUS ao longo do ano:
COMPETENCIA
202401    345455
202402    345710
202403    345590
202404    346544
202405    347422
202406    348208
202407    349101
202408    349226
202409    349738
202410    349923
202411    350333
202412    350540
Name: leitos_sus, dtype: int64


> A série mensal acima é um teste de sanidade importante: leitos SUS variam pouco de um mês para o outro. Um salto abrupto indica competência incompleta ou reprocessamento retroativo, e merece investigação antes de seguir.

## 11. Extração do grupo ST

Uma única competência de referência. O `ST` não entra em nenhum cálculo do índice — ele serve para caracterizar o estabelecimento: tipo de unidade, gestão, esfera administrativa e natureza jurídica.

### O nome do estabelecimento não está aqui

O layout do `ST` tem 208 colunas e **nenhuma delas é razão social ou nome fantasia**. Isso é fácil de errar, porque parece natural que o cadastro de estabelecimentos traga o nome. Não traz: os nomes ficam em arquivos separados, `DBF/CADGER{UF}.dbf`, dentro do mesmo `TAB_CNES.zip` que a seção 7.1 já baixa. O `Leitos_Especialidade.def` declara essa origem explicitamente:

```text
DES Nome Fantasia - SP, CNES, FANTASIA, DBF\CADGERSP.DBF
SES Razao Social  - SP, CNES, RAZ_SOCI, DBF\CADGERSP.DBF
```

Enquanto o `CADGER` não for incorporado, o dashboard identifica hospital por código `CNES`, não por nome. Fica registrado como pendência — são 553 MB descompactados no Brasil inteiro, então vale ler apenas `CNES`, `FANTASIA` e `RAZ_SOCI` e filtrar pelos 9.217 estabelecimentos que têm leito, em vez de carregar tudo.

### As colunas que existem de verdade

Três nomes da versão anterior deste notebook não existem no layout: `RAZAO_SOC`, `NOME_FANT` e `GESTAO`. A coluna de gestão chama-se `TPGESTAO`. Pedir colunas inexistentes não derruba o notebook — o `selecionar_colunas` avisa e segue —, mas produziria uma tabela silenciosamente mais pobre do que o esperado.

In [ ]:
# Colunas conferidas contra o layout real do ST (208 colunas) E contra o
# preenchimento: ESFERA_A, NATUREZA e NIV_HIER existem no layout mas vêm
# 100% vazias nos 445 mil registros, então ficaram de fora. Estavam na
# versão anterior deste notebook ocupando espaço sem informar nada.
COLUNAS_ST = [
    "CNES", "CODUFMUN", "TP_UNID", "TPGESTAO", "NAT_JUR",
    "ATIVIDAD", "NIV_DEP", "TURNO_AT", "VINC_SUS", "CLIENTEL",
]

print(f"Extraindo ST: {len(UFS)} UF(s) x 1 competência ({ANO}{str(MES_ST).zfill(2)})")
cnes_st_bruto = extrair_grupo("ST", UFS, ANO, [MES_ST], colunas=COLUNAS_ST)

cnes_st = selecionar_colunas(cnes_st_bruto, COLUNAS_ST + ["UF", "COMPETENCIA"])

for col in COLUNAS_ST:
    if col in cnes_st.columns:
        cnes_st[col] = cnes_st[col].astype(str).str.strip()
cnes_st["CNES"] = cnes_st["CNES"].str.zfill(7)
cnes_st["CODUFMUN"] = cnes_st["CODUFMUN"].str.zfill(6)

# A natureza jurídica é organizada por grupo no primeiro dígito. Serve de
# rede de segurança: o NATJURC.CNV lista os códigos específicos, mas não os
# códigos de grupo como "4000", que sozinho responde por 26% dos
# estabelecimentos — consultórios individuais.
GRUPO_NAT_JUR = {
    "1": "Administração Pública",
    "2": "Entidade Empresarial",
    "3": "Entidade sem Fins Lucrativos",
    "4": "Pessoa Física",
    "5": "Organização Internacional",
}

# Traduz cada código para texto usando os CNVs da seção 7.1.
for coluna in CNV_ST:
    if coluna not in cnes_st.columns:
        continue
    antes = len(cnes_st)
    cnes_st = cnes_st.merge(dominios[coluna], on=coluna, how="left")
    assert len(cnes_st) == antes, f"o domínio de {coluna} multiplicou linhas"

    if coluna == "NAT_JUR":
        # Sem o CNV específico, ao menos o grupo — melhor que "não catalogado".
        cnes_st[f"{coluna}_DESC"] = cnes_st[f"{coluna}_DESC"].fillna(
            cnes_st[coluna].str[0].map(GRUPO_NAT_JUR)
        )
    cnes_st[f"{coluna}_DESC"] = cnes_st[f"{coluna}_DESC"].fillna("(não catalogado no CNV)")

print(f"\nST: {cnes_st.shape[0]:,} estabelecimentos em {cnes_st['UF'].nunique()} UF(s)")
print(f"Memória: {cnes_st.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

# Código sem tradução é código novo do Ministério ou CNV desatualizado.
# Reportar é melhor que deixar virar texto genérico sem ninguém notar.
print("\nCobertura da tradução:")
for coluna in CNV_ST:
    if coluna not in cnes_st.columns:
        continue
    sem = cnes_st[f"{coluna}_DESC"] == "(não catalogado no CNV)"
    vazios = cnes_st[coluna] == ""
    orfaos = sorted(cnes_st.loc[sem & ~vazios, coluna].unique())
    situacao = "todos traduzidos" if not orfaos else f"sem tradução: {orfaos[:5]}"
    print(f"  {coluna:<10} {situacao}")

# O ST cobre todo estabelecimento cadastrado, não só os que têm leito.
com_leito = set(cnes_lt_filtrado["CNES"])
print(f"\nEstabelecimentos com leito que o ST descreve: "
      f"{cnes_st['CNES'].isin(com_leito).sum():,} de {len(com_leito):,}")

display(cnes_st[["CNES", "TP_UNID", "TP_UNID_DESC", "TPGESTAO_DESC", "NAT_JUR_DESC"]].head())

## 12. Salvamento em Parquet

Os arquivos ficam em `dados/tratado/cnes/`, na raiz do repositório, e seguem a convenção `{fonte}_{grupo}_{uf}_{competência}.parquet`.

**Um arquivo por UF, não um único arquivo do Brasil.** É o que permite o curinga (`cnes_lt_*.parquet`) funcionar quando estes arquivos forem carregados no Autonomous Database, e o que deixa reprocessar um estado sem mexer nos outros. As tabelas Silver, que já são agregadas e pequenas, ficam em arquivo único.

O upload para o Object Storage é manual — o caminho e o tamanho de cada arquivo são impressos abaixo para conferência antes de subir.

In [16]:
arquivos_gerados = []


def salvar(df: pd.DataFrame, nome: str) -> None:
    destino = DIRETORIO_TRATADO / nome
    df.to_parquet(destino, index=False)
    arquivos_gerados.append(destino)


# Um arquivo por UF mantém a convenção de nomes e permite reprocessar
# um estado isolado sem tocar nos demais.
for uf, parte in cnes_lt_filtrado.groupby("UF"):
    salvar(parte, f"cnes_lt_{uf}_{ANO}.parquet")

if not cnes_st.empty:
    for uf, parte in cnes_st.groupby("UF"):
        salvar(parte, f"cnes_st_{uf}_{ANO}{str(MES_ST).zfill(2)}.parquet")

# As Silver já são agregadas e pequenas — arquivo único é mais simples de consumir.
salvar(silver_cnes_municipio, f"cnes_silver_municipio_{ANO}.parquet")
salvar(silver_cnes_hospital, f"cnes_silver_hospital_{ANO}.parquet")

total_mb = sum(a.stat().st_size for a in arquivos_gerados) / 1024**2
print(f"{len(arquivos_gerados)} arquivos, {total_mb:.1f} MB")
print(f"Pasta: {DIRETORIO_TRATADO}\n")

for arq in sorted(arquivos_gerados):
    print(f"  {arq.name:<42} {arq.stat().st_size / 1024:>9,.1f} KB")

56 arquivos, 7.6 MB
Pasta: C:\Users\Vitor Nobre\Documents\Workspace\conecta-saude\dados\tratado\cnes

  cnes_lt_AC_2024.parquet                         21.4 KB
  cnes_lt_AL_2024.parquet                         45.3 KB
  cnes_lt_AM_2024.parquet                         47.3 KB
  cnes_lt_AP_2024.parquet                         15.3 KB
  cnes_lt_BA_2024.parquet                        225.1 KB
  cnes_lt_CE_2024.parquet                        121.8 KB
  cnes_lt_DF_2024.parquet                         33.6 KB
  cnes_lt_ES_2024.parquet                         56.8 KB
  cnes_lt_GO_2024.parquet                        161.0 KB
  cnes_lt_MA_2024.parquet                         95.8 KB
  cnes_lt_MG_2024.parquet                        296.4 KB
  cnes_lt_MS_2024.parquet                         54.9 KB
  cnes_lt_MT_2024.parquet                         71.3 KB
  cnes_lt_PA_2024.parquet                        111.3 KB
  cnes_lt_PB_2024.parquet                         67.3 KB
  cnes_lt_PE_2024.parquet   

## 13. Perguntas que o CNES responde

1. Quantos leitos SUS existem em cada município?
2. Quantos estabelecimentos com leito cada município possui?
3. Quais municípios não têm nenhum leito SUS — os vazios assistenciais?
4. Quais hospitais concentram a maior oferta de leitos?
5. Como a oferta de leitos evoluiu ao longo das 12 competências?
6. Qual a proporção entre leitos existentes e leitos disponíveis ao SUS?

Rodando o Brasil inteiro, a pergunta 3 ganha peso: o `LT` cobre 3.619 municípios, contra os 5.570 existentes. A diferença não é falha de extração — é a informação em si, e são esses quase 2 mil municípios sem nenhum leito cadastrado que o índice precisa enxergar.

Cruzado com SIH/SUS e IBGE:

```text
Internações por leito SUS      = internações SIH / leitos SUS CNES
Leitos SUS por 10 mil hab.     = (leitos SUS CNES / população IBGE) * 10.000
```

Essas duas métricas separam o município que tem volume alto por ser grande daquele que tem pressão real sobre uma estrutura insuficiente — que é a tese central do Conecta Saúde.

## 14. Próximo passo

Com o `silver_cnes_municipio` gerado, o CNES está pronto. Os arquivos ficam em `dados/tratado/cnes/` e o envio ao Object Storage é feito manualmente, fora deste notebook.

Etapas seguintes:

1. extrair o SIH/SUS, que é a fonte crítica e o numerador do índice — lá o terminador quebrado do DBF reaparece, já que a falha não é exclusiva do CNES;
2. subir os Parquet ao bucket e criar as tabelas externas Bronze no Autonomous Database;
3. incorporar o `CADGER` para dar nome aos estabelecimentos, conforme registrado na seção 11.

O projeto usa quatro fontes: CNES, IBGE, SIGTAP e SIH/SUS. O SIA/SUS e a regulação foram avaliados e ficaram fora do escopo — a justificativa está no topo dos respectivos notebooks.